In [1]:
!pip install openpyxl

In [2]:
import pandas as pd

In [3]:
stats = pd.read_excel(r"C:\Users\gilmo\Documents\MLS STATS 2025.xlsx")

In [5]:
# set the real column names
stats.columns = stats.iloc[0]

# drop the first row (which contained the headers)
stats = stats.drop(0)

# reset index
stats = stats.reset_index(drop=True)

In [6]:
stats.head()

,Rk,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,...,PK,PKatt,CrdY,CrdR,Gls,Ast,G+A,G-PK,G+A-PK,Matches
0,1,Paxten Aaronson,us USA,MF,Colorado Rapids,21,2003,7,6,572,...,0,0,3,0,0.16,0,0.16,0.16,0.16,Matches
1,2,Liel Abada,il ISR,"MF,FW",Charlotte,23,2001,32,17,1481,...,0,0,1,0,0.3,0.06,0.36,0.3,0.36,Matches
2,3,Wessam Abou Ali,ps PLE,FW,Columbus Crew,26,1999,5,4,303,...,0,0,0,0,0.89,0,0.89,0.89,0.89,Matches
3,4,Luis Abram,pe PER,DF,Atlanta Utd,28,1996,21,16,1493,...,0,0,1,0,0,0,0,0,0,Matches
4,5,Lalas Abubakar,gh GHA,DF,FC Dallas,30,1994,29,18,1656,...,0,0,5,1,0.05,0,0.05,0.05,0.05,Matches


In [7]:
stats.shape

(917, 25)

In [8]:
!pip install tabula-py pandas openpyxl

In [9]:
import pandas as pd

salary = pd.read_excel(r"C:\Users\gilmo\Downloads\mls_salary_2025.xlsx")

salary.head()

,First Name,Last Name,Team Name,Position,PA Base Salary,Guaranteed Comp
0,Aleksey,Miranchuk,Atlanta United,Attacking Midfield,3600000.0,4885441
1,Emmanuel,Latte Lath,Atlanta United,Center Forward,3534546.0,4030546
2,Jamal,Thiaré,Atlanta United,Center Forward,700000.0,730000
3,Cayman,Togashi,Atlanta United,Center Forward,104000.0,104000
4,Enea,Mihaj,Atlanta United,Center-back,1030000.0,1372000


In [10]:
salary.shape

(944, 6)

In [11]:
print(stats.columns)

Index(['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Age', 'Born', 'MP', 'Starts',
       'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY',
       'CrdR', 'Gls', 'Ast', 'G+A', 'G-PK', 'G+A-PK', 'Matches'],
      dtype='object', name=0)


In [12]:
print(salary.columns)

Index(['First Name', 'Last Name', 'Team Name', 'Position', 'PA Base Salary',
       'Guaranteed Comp'],
      dtype='object')


In [18]:
stats["player"] = stats["player"].replace({
    "some fbref name": "matching salary name"
})

In [19]:
salary["club"] = salary["club"].replace({
    "some salary club name": "matching stats club name"
})

In [20]:
import pandas as pd
import numpy as np

# -----------------------------
# CLEAN STATS DATASET
# -----------------------------

# remove duplicate column names, keep first occurrence
stats = stats.loc[:, ~stats.columns.duplicated()].copy()

# rename columns
stats = stats.rename(columns={
    "Player": "player",
    "Squad": "club",
    "Pos": "position",
    "Age": "age",
    "MP": "matches_played",
    "Starts": "starts",
    "Min": "minutes",
    "Gls": "goals",
    "Ast": "assists"
})

# keep only needed columns
stats = stats[[
    "player", "club", "position", "age",
    "matches_played", "starts", "minutes", "goals", "assists"
]].copy()

# clean text fields
for col in ["player", "club", "position"]:
    stats[col] = stats[col].astype(str).str.strip()

# convert numeric fields
for col in ["age", "matches_played", "starts", "minutes", "goals", "assists"]:
    stats[col] = pd.to_numeric(stats[col], errors="coerce")

# drop empty player rows
stats = stats.dropna(subset=["player"])

print("Cleaned stats preview:")
print(stats.head())


# -----------------------------
# CLEAN SALARY DATASET
# -----------------------------

salary = salary.rename(columns={
    "First Name": "first_name",
    "Last Name": "last_name",
    "Team Name": "club",
    "Position": "position",
    "PA Base Salary": "base_salary",
    "Guaranteed Comp": "guaranteed_comp"
})

# create full player name
salary["player"] = (
    salary["first_name"].astype(str).str.strip() + " " +
    salary["last_name"].astype(str).str.strip()
)

# keep only needed columns
salary = salary[[
    "player", "club", "position", "base_salary", "guaranteed_comp"
]].copy()

# clean text fields
for col in ["player", "club", "position"]:
    salary[col] = salary[col].astype(str).str.strip()

# clean salary fields
salary["base_salary"] = (
    salary["base_salary"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

salary["guaranteed_comp"] = (
    salary["guaranteed_comp"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

# convert salary fields to numeric
salary["base_salary"] = pd.to_numeric(salary["base_salary"], errors="coerce")
salary["guaranteed_comp"] = pd.to_numeric(salary["guaranteed_comp"], errors="coerce")

print("\nCleaned salary preview:")
print(salary.head())


# -----------------------------
# STANDARDIZE TEXT FOR MERGING
# -----------------------------

def clean_text(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower()

for col in ["player", "club", "position"]:
    stats[col] = stats[col].apply(clean_text)
    salary[col] = salary[col].apply(clean_text)

# optional: standardize club names if needed
club_map = {
    "fc dallas": "dallas",
    "sporting kansas city": "sporting kc",
    "new york city fc": "new york city",
    "los angeles football club": "los angeles fc",
    "la galaxy": "los angeles galaxy"
}

stats["club"] = stats["club"].replace(club_map)
salary["club"] = salary["club"].replace(club_map)

# remove duplicates
stats = stats.drop_duplicates(subset=["player", "club"])
salary = salary.drop_duplicates(subset=["player", "club"])

print("\nStats shape:", stats.shape)
print("Salary shape:", salary.shape)


# -----------------------------
# MERGE DATASETS
# -----------------------------

merged = pd.merge(
    stats,
    salary,
    on=["player", "club"],
    how="inner"
)

print("\nMerged shape:", merged.shape)
print(merged.head())


# -----------------------------
# FEATURE ENGINEERING
# -----------------------------

merged["goal_contributions"] = merged["goals"] + merged["assists"]

merged["goals_per_90"] = np.where(
    merged["minutes"] > 0,
    merged["goals"] / (merged["minutes"] / 90),
    np.nan
)

merged["assists_per_90"] = np.where(
    merged["minutes"] > 0,
    merged["assists"] / (merged["minutes"] / 90),
    np.nan
)

merged["salary_per_goal_contribution"] = np.where(
    merged["goal_contributions"] > 0,
    merged["guaranteed_comp"] / merged["goal_contributions"],
    np.nan
)

print("\nMerged with new variables:")
print(merged.head())

# save cleaned merged dataset
merged.to_csv("mls_merged_cleaned.csv", index=False)

Cleaned stats preview:
0           player             club position   age  matches_played  starts  \
0  paxten aaronson  colorado rapids       mf  21.0             7.0     6.0   
1       liel abada        charlotte    mf,fw  23.0            32.0    17.0   
2  wessam abou ali    columbus crew       fw  26.0             5.0     4.0   
3       luis abram      atlanta utd       df  28.0            21.0    16.0   
4   lalas abubakar           dallas       df  30.0            29.0    18.0   

0  minutes  goals  assists  
0    572.0    1.0      0.0  
1   1481.0    5.0      1.0  
2    303.0    3.0      0.0  
3   1493.0    0.0      0.0  
4   1656.0    1.0      0.0  


KeyError: 'first_name'

In [21]:
merged.shape

(358, 16)

In [22]:
print("Stats players:", len(stats))
print("Salary players:", len(salary))
print("Merged players:", len(merged))

Stats players: 883
Salary players: 944
Merged players: 358


In [23]:
stats_unmatched = stats.merge(salary, on=["player", "club"], how="left", indicator=True)
stats_unmatched = stats_unmatched[stats_unmatched["_merge"] == "left_only"]

salary_unmatched = salary.merge(stats, on=["player", "club"], how="left", indicator=True)
salary_unmatched = salary_unmatched[salary_unmatched["_merge"] == "left_only"]

print("Unmatched stats:")
print(stats_unmatched[["player", "club"]].head(20))

print("\nUnmatched salary:")
print(salary_unmatched[["player", "club"]].head(20))

Unmatched stats:
                    player              club
1               liel abada         charlotte
3               luis abram       atlanta utd
7           luciano acosta            dallas
8             sam adekugbe  vancouver w'caps
9          leonardo afonso       atlanta utd
10         leonardo afonso       inter miami
12           william agada       sporting kc
13           william agada    real salt lake
14        patrick agyemang         charlotte
15               ali ahmed  vancouver w'caps
20              rasmus alm    st. louis city
21          miguel almirón       atlanta utd
22  alejandro alvarado jr.      san diego fc
23        fernando álvarez       cf montréal
24           steven alzate       atlanta utd
25                  player             squad
26            pedro amador       atlanta utd
31          felipe andrade    houston dynamo
32             tomas ángel      san diego fc
33             iván angulo      orlando city

Unmatched salary:
                 pl

In [24]:
# see current columns
print("Current salary columns:", salary.columns.tolist())

# rename only if original names still exist
salary = salary.rename(columns={
    "First Name": "first_name",
    "Last Name": "last_name",
    "Team Name": "club",
    "Position": "position",
    "PA Base Salary": "base_salary",
    "Guaranteed Comp": "guaranteed_comp"
})

print("After possible rename:", salary.columns.tolist())

# create player column only if it does not already exist
if "player" not in salary.columns:
    salary["player"] = (
        salary["first_name"].astype(str).str.strip() + " " +
        salary["last_name"].astype(str).str.strip()
    )

# keep only needed columns that exist
needed_salary = ["player", "club", "position", "base_salary", "guaranteed_comp"]
salary = salary[[col for col in needed_salary if col in salary.columns]].copy()

# clean text
for col in ["player", "club", "position"]:
    if col in salary.columns:
        salary[col] = salary[col].astype(str).str.strip()

# clean salary numbers
for col in ["base_salary", "guaranteed_comp"]:
    if col in salary.columns:
        salary[col] = (
            salary[col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        salary[col] = pd.to_numeric(salary[col], errors="coerce")

print(salary.head())

Current salary columns: ['player', 'club', 'position', 'base_salary', 'guaranteed_comp']
After possible rename: ['player', 'club', 'position', 'base_salary', 'guaranteed_comp']
                player            club            position  base_salary  \
0    aleksey miranchuk  atlanta united  attacking midfield    3600000.0   
1  emmanuel latte lath  atlanta united      center forward    3534546.0   
2         jamal thiaré  atlanta united      center forward     700000.0   
3       cayman togashi  atlanta united      center forward     104000.0   
4           enea mihaj  atlanta united         center-back    1030000.0   

   guaranteed_comp  
0          4885441  
1          4030546  
2           730000  
3           104000  
4          1372000  


In [25]:
print("Current stats columns:", stats.columns.tolist())

# remove duplicate columns
stats = stats.loc[:, ~stats.columns.duplicated()].copy()

# rename only if original names still exist
stats = stats.rename(columns={
    "Player": "player",
    "Squad": "club",
    "Pos": "position",
    "Age": "age",
    "MP": "matches_played",
    "Starts": "starts",
    "Min": "minutes",
    "Gls": "goals",
    "Ast": "assists"
})

needed_stats = [
    "player", "club", "position", "age",
    "matches_played", "starts", "minutes", "goals", "assists"
]

stats = stats[[col for col in needed_stats if col in stats.columns]].copy()

for col in ["player", "club", "position"]:
    if col in stats.columns:
        stats[col] = stats[col].astype(str).str.strip()

for col in ["age", "matches_played", "starts", "minutes", "goals", "assists"]:
    if col in stats.columns:
        stats[col] = pd.to_numeric(stats[col], errors="coerce")

print(stats.head())

Current stats columns: ['player', 'club', 'position', 'age', 'matches_played', 'starts', 'minutes', 'goals', 'assists']
0           player             club position   age  matches_played  starts  \
0  paxten aaronson  colorado rapids       mf  21.0             7.0     6.0   
1       liel abada        charlotte    mf,fw  23.0            32.0    17.0   
2  wessam abou ali    columbus crew       fw  26.0             5.0     4.0   
3       luis abram      atlanta utd       df  28.0            21.0    16.0   
4   lalas abubakar           dallas       df  30.0            29.0    18.0   

0  minutes  goals  assists  
0    572.0    1.0      0.0  
1   1481.0    5.0      1.0  
2    303.0    3.0      0.0  
3   1493.0    0.0      0.0  
4   1656.0    1.0      0.0  


In [27]:
import numpy as np

def clean_text(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower()

for col in ["player", "club", "position"]:
    if col in stats.columns:
        stats[col] = stats[col].apply(clean_text)
    if col in salary.columns:
        salary[col] = salary[col].apply(clean_text)

# optional team-name mapping
club_map = {
    "fc dallas": "dallas",
    "sporting kansas city": "sporting kc",
    "new york city fc": "new york city",
    "los angeles football club": "los angeles fc",
    "la galaxy": "los angeles galaxy"
}

stats["club"] = stats["club"].replace(club_map)
salary["club"] = salary["club"].replace(club_map)

stats = stats.drop_duplicates(subset=["player", "club"])
salary = salary.drop_duplicates(subset=["player", "club"])

merged_new = pd.merge(stats, salary, on=["player", "club"], how="inner")

print("Stats shape:", stats.shape)
print("Salary shape:", salary.shape)
print("Merged New shape:", merged_new.shape)

merged_new.head()

Stats shape: (883, 9)
Salary shape: (944, 5)
Merged New shape: (358, 12)


,player,club,position_x,age,matches_played,starts,minutes,goals,assists,position_y,base_salary,guaranteed_comp
0,paxten aaronson,colorado rapids,mf,21.0,7.0,6.0,572.0,1.0,0.0,attacking midfield,2000000.0,2228063
1,wessam abou ali,columbus crew,fw,26.0,5.0,4.0,303.0,3.0,0.0,center forward,1800000.0,2157375
2,lalas abubakar,dallas,df,30.0,29.0,18.0,1656.0,1.0,0.0,center-back,410000.0,445000
3,bryan acosta,nashville sc,mf,31.0,14.0,1.0,221.0,0.0,0.0,central midfield,175000.0,196250
4,kellyn acosta,chicago fire,mf,29.0,19.0,6.0,655.0,0.0,0.0,defensive midfield,1461796.0,1628477


In [28]:
print("Stats clubs:")
print(sorted(stats["club"].dropna().unique()))

print("\nSalary clubs:")
print(sorted(salary["club"].dropna().unique()))

Stats clubs:
['atlanta utd', 'austin fc', 'cf montréal', 'charlotte', 'chicago fire', 'colorado rapids', 'columbus crew', 'd.c. united', 'dallas', 'fc cincinnati', 'houston dynamo', 'inter miami', 'lafc', 'los angeles galaxy', 'minnesota utd', 'nashville sc', 'ne revolution', 'ny red bulls', 'nycfc', 'orlando city', 'philadelphia union', 'portland timbers', 'real salt lake', 'san diego fc', 'seattle sounders', 'sj earthquakes', 'sporting kc', 'squad', 'st. louis city', 'toronto fc', "vancouver w'caps"]

Salary clubs:
['atlanta united', 'austin fc', 'cf montreal', 'charlotte fc', 'chicago fire', 'colorado rapids', 'columbus crew', 'dallas', 'dc united', 'fc cincinnati', 'houston dynamo', 'inter miami', 'lafc', 'los angeles galaxy', 'minnesota united', 'mls pool', 'nashville sc', 'new england revolutio', 'new york city', 'new york red bulls', 'orlando city sc', 'philadelphia union', 'portland timbers', 'real salt lake', 'retired', 'san diego fc', 'san jose earthquakes', 'seattle sounders

In [29]:
stats = stats[stats["club"] != "squad"]

In [30]:
salary = salary[~salary["club"].isin(["mls pool", "retired", "without a club"])]

In [31]:
club_map = {
    # Montreal
    "cf montréal": "cf montreal",
    
    # DC United
    "d.c. united": "dc united",
    
    # NY teams
    "ny red bulls": "new york red bulls",
    "nycfc": "new york city",
    
    # New England
    "ne revolution": "new england revolutio",
    
    # Minnesota
    "minnesota utd": "minnesota united",
    
    # San Jose
    "sj earthquakes": "san jose earthquakes",
    
    # Seattle
    "seattle sounders": "seattle sounders fc",
    
    # St Louis
    "st. louis city": "st. louis city sc",
    
    # Vancouver
    "vancouver w'caps": "vancouver whitecaps",
    
    # Charlotte
    "charlotte": "charlotte fc",
    
    # Orlando
    "orlando city": "orlando city sc",
    
    # Atlanta
    "atlanta utd": "atlanta united"
}

In [32]:
stats["club"] = stats["club"].replace(club_map)
salary["club"] = salary["club"].replace(club_map)

In [33]:
def clean_text(x):
    return str(x).strip().lower()

stats["club"] = stats["club"].apply(clean_text)
salary["club"] = salary["club"].apply(clean_text)

In [34]:
merged_new = pd.merge(
    stats,
    salary,
    on=["player", "club"],
    how="inner"
)

print("Stats:", stats.shape)
print("Salary:", salary.shape)
print("Merged:", merged_new.shape)

Stats: (882, 9)
Salary: (931, 5)
Merged: (645, 12)


In [35]:
import re
import unicodedata
import numpy as np
import pandas as pd

def clean_name(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    x = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", x)
    x = re.sub(r"[^a-z0-9\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

stats["player_clean"] = stats["player"].apply(clean_name)
salary["player_clean"] = salary["player"].apply(clean_name)

stats["club_clean"] = stats["club"].apply(clean_name)
salary["club_clean"] = salary["club"].apply(clean_name)

In [36]:
merged_names = pd.merge(
    stats,
    salary,
    on=["player_clean", "club_clean"],
    how="inner",
    suffixes=("_stats", "_salary")
)

print("Merged shape after cleaned names:", merged_names.shape)
merged_names.head()

Merged shape after cleaned names: (678, 16)


,player_stats,club_stats,position_stats,age,matches_played,starts,minutes,goals,assists,player_clean,club_clean,player_salary,club_salary,position_salary,base_salary,guaranteed_comp
0,paxten aaronson,colorado rapids,mf,21.0,7.0,6.0,572.0,1.0,0.0,paxten aaronson,colorado rapids,paxten aaronson,colorado rapids,attacking midfield,2000000.0,2228063
1,liel abada,charlotte fc,"mf,fw",23.0,32.0,17.0,1481.0,5.0,1.0,liel abada,charlotte fc,liel abada,charlotte fc,right wing,2400000.0,2548500
2,wessam abou ali,columbus crew,fw,26.0,5.0,4.0,303.0,3.0,0.0,wessam abou ali,columbus crew,wessam abou ali,columbus crew,center forward,1800000.0,2157375
3,lalas abubakar,dallas,df,30.0,29.0,18.0,1656.0,1.0,0.0,lalas abubakar,dallas,lalas abubakar,dallas,center-back,410000.0,445000
4,bryan acosta,nashville sc,mf,31.0,14.0,1.0,221.0,0.0,0.0,bryan acosta,nashville sc,bryan acosta,nashville sc,central midfield,175000.0,196250


In [38]:
stats_unmatched = stats.merge(
    salary,
    on=["player_clean", "club_clean"],
    how="left",
    indicator=True,
    suffixes=("_stats", "_salary")
)

stats_unmatched = stats_unmatched[stats_unmatched["_merge"] == "left_only"]

print(stats_unmatched[["player_stats", "club_stats"]].drop_duplicates().head(30))

              player_stats             club_stats
3               luis abram         atlanta united
7           luciano acosta                 dallas
9          leonardo afonso         atlanta united
10         leonardo afonso            inter miami
12           william agada            sporting kc
13           william agada         real salt lake
14        patrick agyemang           charlotte fc
30          felipe andrade         houston dynamo
34                  antony       portland timbers
36            brian anunga          fc cincinnati
37        chris applewhite           nashville sc
41     daniel armando ríos    vancouver whitecaps
43                   artur         houston dynamo
44          joshua atencio        colorado rapids
48        chidozie awaziem        colorado rapids
49         obafemi awodesu         houston dynamo
54             corey baird          fc cincinnati
57           monsef bakrar          new york city
60           fidel barajas              dc united


In [39]:
salary_unmatched = salary.merge(
    stats,
    on=["player_clean", "club_clean"],
    how="left",
    indicator=True,
    suffixes=("_salary", "_stats")
)

salary_unmatched = salary_unmatched[salary_unmatched["_merge"] == "left_only"]

print(salary_unmatched[["player_salary", "club_salary"]].drop_duplicates().head(30))

                   player_salary     club_salary
0              aleksey miranchuk  atlanta united
6                stian gregersen  atlanta united
8                    will reilly  atlanta united
14                   adyn torres  atlanta united
15                    josh cohen  atlanta united
19                    leo afonso  atlanta united
20               saba lobjanidze  atlanta united
25                 ashton gordon  atlanta united
27                   nyk sessock  atlanta united
30                  micah burton       austin fc
36             mateja djordjević       austin fc
42              stefan cleveland       austin fc
44                    damian las       austin fc
48   guilherme da trindade dubas       austin fc
51               jimmy farkarlun       austin fc
55                  riley thomas       austin fc
58                    iván jaime     cf montreal
59                matías cóccaro     cf montreal
61            owen graham-roache     cf montreal
68               mat

In [40]:
merged_names

,player_stats,club_stats,position_stats,age,matches_played,starts,minutes,goals,assists,player_clean,club_clean,player_salary,club_salary,position_salary,base_salary,guaranteed_comp
0,paxten aaronson,colorado rapids,mf,21.0,7.0,6.0,572.0,1.0,0.0,paxten aaronson,colorado rapids,paxten aaronson,colorado rapids,attacking midfield,2000000.0,2228063
1,liel abada,charlotte fc,"mf,fw",23.0,32.0,17.0,1481.0,5.0,1.0,liel abada,charlotte fc,liel abada,charlotte fc,right wing,2400000.0,2548500
2,wessam abou ali,columbus crew,fw,26.0,5.0,4.0,303.0,3.0,0.0,wessam abou ali,columbus crew,wessam abou ali,columbus crew,center forward,1800000.0,2157375
3,lalas abubakar,dallas,df,30.0,29.0,18.0,1656.0,1.0,0.0,lalas abubakar,dallas,lalas abubakar,dallas,center-back,410000.0,445000
4,bryan acosta,nashville sc,mf,31.0,14.0,1.0,221.0,0.0,0.0,bryan acosta,nashville sc,bryan acosta,nashville sc,central midfield,175000.0,196250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673,sean zawadzki,columbus crew,"df,mf",24.0,27.0,27.0,2430.0,2.0,0.0,sean zawadzki,columbus crew,sean zawadzki,columbus crew,central midfield,400000.0,446875
674,walker zimmerman,nashville sc,df,31.0,21.0,16.0,1532.0,0.0,2.0,walker zimmerman,nashville sc,walker zimmerman,nashville sc,center-back,3200000.0,3456979
675,philip zinckernagel,chicago fire,"fw,mf",30.0,32.0,31.0,2516.0,15.0,12.0,philip zinckernagel,chicago fire,philip zinckernagel,chicago fire,right wing,1450750.0,1631290
676,rida zouhir,dc united,mf,21.0,15.0,2.0,335.0,0.0,0.0,rida zouhir,dc united,rida zouhir,dc united,central midfield,80622.0,95362


In [41]:
salary.shape

(931, 7)

In [42]:
stats.shape

(882, 11)

In [43]:
stats = stats.sort_values("minutes", ascending=False)
stats = stats.drop_duplicates(subset=["player_clean"])

In [44]:
merged_final = pd.merge(
    stats,
    salary,
    on=["player_clean", "club_clean"],
    how="inner",
    suffixes=("_stats", "_salary")
)

print("Final merged shape:", merged_final.shape)

Final merged shape: (664, 16)


In [45]:
stats

,player,club,position,age,matches_played,starts,minutes,goals,assists,player_clean,club_clean
135,rafael cabral,real salt lake,gk,34.0,34.0,34.0,3060.0,0.0,0.0,rafael cabral,real salt lake
793,brad stuver,austin fc,gk,33.0,34.0,34.0,3060.0,0.0,0.0,brad stuver,austin fc
661,john pulskamp,sporting kc,gk,23.0,34.0,34.0,3060.0,0.0,2.0,john pulskamp,sporting kc
808,yohei takaoka,vancouver whitecaps,gk,28.0,34.0,34.0,3060.0,0.0,0.0,yohei takaoka,vancouver whitecaps
888,joe willis,nashville sc,gk,36.0,34.0,34.0,3060.0,0.0,0.0,joe willis,nashville sc
...,...,...,...,...,...,...,...,...,...,...,...
874,sydney wathuta,colorado rapids,df,20.0,1.0,0.0,1.0,0.0,0.0,sydney wathuta,colorado rapids
470,antino lopez,seattle sounders fc,df,22.0,1.0,0.0,1.0,0.0,0.0,antino lopez,seattle sounders fc
371,luke hille,minnesota united,mf,21.0,1.0,0.0,1.0,0.0,0.0,luke hille,minnesota united
152,jackson castro,vancouver whitecaps,mf,22.0,1.0,0.0,1.0,0.0,0.0,jackson castro,vancouver whitecaps


In [46]:
salary

,player,club,position,base_salary,guaranteed_comp,player_clean,club_clean
0,aleksey miranchuk,atlanta united,attacking midfield,3600000.0,4885441,aleksey miranchuk,atlanta united
1,emmanuel latte lath,atlanta united,center forward,3534546.0,4030546,emmanuel latte lath,atlanta united
2,jamal thiaré,atlanta united,center forward,700000.0,730000,jamal thiare,atlanta united
3,cayman togashi,atlanta united,center forward,104000.0,104000,cayman togashi,atlanta united
4,enea mihaj,atlanta united,center-back,1030000.0,1372000,enea mihaj,atlanta united
...,...,...,...,...,...,...,...
938,tate johnson,vancouver whitecaps,left-back,80622.0,81622,tate johnson,vancouver whitecaps
939,sam adekugbe,vancouver whitecaps,left-back,875000.0,963000,sam adekugbe,vancouver whitecaps
940,emmanuel sabbi,vancouver whitecaps,right wing,900000.0,987750,emmanuel sabbi,vancouver whitecaps
941,giuseppe bovalina,vancouver whitecaps,right-back,104000.0,116980,giuseppe bovalina,vancouver whitecaps


In [47]:
merged_final

,player_stats,club_stats,position_stats,age,matches_played,starts,minutes,goals,assists,player_clean,club_clean,player_salary,club_salary,position_salary,base_salary,guaranteed_comp
0,brad stuver,austin fc,gk,33.0,34.0,34.0,3060.0,0.0,0.0,brad stuver,austin fc,brad stuver,austin fc,goalkeeper,484500.0,507313
1,john pulskamp,sporting kc,gk,23.0,34.0,34.0,3060.0,0.0,2.0,john pulskamp,sporting kc,john pulskamp,sporting kc,goalkeeper,250000.0,273500
2,yohei takaoka,vancouver whitecaps,gk,28.0,34.0,34.0,3060.0,0.0,0.0,yohei takaoka,vancouver whitecaps,yohei takaoka,vancouver whitecaps,goalkeeper,756000.0,873713
3,joe willis,nashville sc,gk,36.0,34.0,34.0,3060.0,0.0,0.0,joe willis,nashville sc,joe willis,nashville sc,goalkeeper,600000.0,658333
4,carles gil,new england revolutio,"mf,fw",32.0,34.0,34.0,3054.0,10.0,9.0,carles gil,new england revolutio,carles gil,new england revolutio,attacking midfield,4250000.0,4702083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
659,jack neeley,charlotte fc,df,19.0,1.0,0.0,3.0,0.0,0.0,jack neeley,charlotte fc,jack neeley,charlotte fc,center-back,104000.0,106000
660,eric klein,new england revolutio,mf,18.0,2.0,0.0,2.0,0.0,0.0,eric klein,new england revolutio,eric klein,new england revolutio,center-back,80622.0,86899
661,diadie samassékou,houston dynamo,mf,29.0,1.0,0.0,1.0,0.0,0.0,diadie samassekou,houston dynamo,diadié samassékou,houston dynamo,defensive midfield,714996.0,771871
662,anthony ramirez,dallas,fw,19.0,1.0,0.0,1.0,0.0,0.0,anthony ramirez,dallas,anthony ramírez,dallas,right wing,80622.0,80622
